Bu notebook, yarışma verisinin kullanımı ve basit bir word co-occurrence mantığı ile tahmin yapılmasını içermektedir.

Notebooktaki örnek kod; sorgu ile ürün **en az bir ortak kelimeyi** paylaşıyorsa **alakalı (1)**, paylaşmıyorsa **alakasız (0)** olarak etiketleme yaklaşımını simüle etmektedir.

In [1]:
from pathlib import Path

DATA_DIR = Path("/kaggle/input/competitions/trendyol-e-ticaret-yarismasi-2026-kaggle")

MIN_TOKEN_LEN = 2   # bu uzunluktan kısa kelimeleri yok say (tek karakterleri eler)
MIN_OVERLAP   = 1   # 'relevant' demek için gereken ortak kelime sayısı

assert (DATA_DIR / "submission_pairs.csv").exists(), f"veri bulunamadı: {DATA_DIR.resolve()}"

In [2]:
import pandas as pd

pairs = pd.read_csv(DATA_DIR / "submission_pairs.csv")   # id, term_id, item_id
terms = pd.read_csv(DATA_DIR / "terms.csv")             # term_id, query
items = pd.read_csv(DATA_DIR / "items.csv")             # item_id, title, category, ...

test = pairs.merge(terms, on="term_id", how="left").merge(items, on="item_id", how="left")
print(f"{len(test):,} çift yüklendi")
test.sample(5, random_state=42)

3,359,679 çift yüklendi


,id,term_id,item_id,query,title,category,brand,gender,age_group,attributes
1160283,TST_104765a215fac0,TERM_1d855961,ITEM_7c79134be0e7,ahşap top kapak,dekoratif kırmızı elma set,ev & mobilya/ev/ev dekorasyon/dekoratif obje v...,hesabınca,unknown,unknown,"materyal: polyester, renk: kırmızı, boyut/ebat..."
2253359,TST_1d98bfc10f53f9,TERM_7d8e6066,ITEM_c59aa11af10b,erkek çocuk outdoor bot,weather forecast wf çelik uçlu mavi kedi köpek...,süpermarket/pet shop/kedi ürünleri/kedi makası,life petmarket,unknown,unknown,"menşei: tr, bakım talimatları (genel): ürünün ..."
2799740,TST_9dc3efbda809ab,TERM_4573472a,ITEM_52c14f939194,çiğköfte baharatı,"leila kiremit renk yatak odası, mutfak, yemek ...",ev & mobilya/mobilya/elektrik & aydınlatma/avize,decory,unknown,unknown,"materyal: plastik, model: modern, duy tipi: e2..."
568555,TST_ae702c8e9ce31f,TERM_b9efbf1d,ITEM_30c55ec44b37,shea butter,şampuan ve saç bakım kremi kepeğe karşı etkili...,kozmetik & kişisel bakım/saç bakım/şampuan,elidor,kadın,yetişkin,"özellik: sülfatsız, etki: kepek önleyici, haci..."
2493674,TST_47d25c75035b23,TERM_0b12f7c6,ITEM_71f3fc04574f,erkek çocuk paten,kartela kanatlı at kız çocuk erkek çocuk oyunc...,anne & bebek & çocuk/oyuncak/figür oyuncaklar/...,paraply,unisex,bebek & çocuk,"paket içeriği: 1'li, renk: karışık, yaş: 1+ ya..."


In [3]:
import re
import numpy as np

_TOKEN_SPLIT_RE = re.compile(r"[^0-9a-zçğıöşü]+")

def _norm(s):
    if not isinstance(s, str):
        return ""
    s = s.replace("İ", "i").replace("I", "ı")
    return s.lower().replace("i̇", "i")

def tokenize(s, min_len=MIN_TOKEN_LEN):
    return {t for t in _TOKEN_SPLIT_RE.split(_norm(s)) if len(t) >= min_len}

queries = test["query"].tolist()

# Ürün metni: başlık + kategori + marka + cinsiyet + yaş grubu + özellikler
item_texts = (
    test["title"].fillna("") + " " + test["category"].fillna("") + " "
    + test["brand"].fillna("") + " " + test["gender"].fillna("") + " "
    + test["age_group"].fillna("") + " " + test["attributes"].fillna("")
).tolist()

n = len(queries)
overlap = np.empty(n, dtype=np.int32)
for i in range(n):
    q = tokenize(queries[i])
    overlap[i] = len(q & tokenize(item_texts[i])) if q else 0

# Ortak kelime sayısı eşiği geçiyorsa 'relevant' olarak işaretlenir
pred = (overlap >= MIN_OVERLAP).astype(np.int8)

In [4]:
submission = pd.DataFrame({"id": test["id"].values, "prediction": pred})
submission.to_csv("submission.csv", index=False)